# 🌙→☀️ Night→Day CycleGAN Training

**Projet:** ESIEA Embedded AI - Translation Night→Day  
**Dataset:** huggan/night2day (HuggingFace)  
**Modèle:** CycleGAN (ResNet-6 blocks)

---

## 📋 Ce que fait ce notebook

1. ✅ Clone ton repo GitHub
2. ✅ Installe les dépendances
3. ✅ Teste le dataset
4. ✅ Entraîne CycleGAN (50-100 epochs)
5. ✅ Génère des samples
6. ✅ Télécharge le checkpoint

**Temps estimé:** 4-6h (selon GPU)

---

## ⚙️ Configuration GPU

**Important:** Aller dans `Runtime` → `Change runtime type` → `Hardware accelerator` → **GPU (T4)**

## 1️⃣ Setup - Clone du repo et installation

In [ ]:
# Clone ton repo GitHub
!git clone https://github.com/pierre265/Night-Day-translation-unpaired-CycleGAN-pix2pix-style-.git
%cd Night-Day-translation-unpaired-CycleGAN-pix2pix-style-

In [ ]:
# Installation des dépendances
!pip install -q datasets huggingface_hub
!pip install -q PyYAML tqdm

print("✅ Dépendances installées")

In [ ]:
# Vérifier GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ PAS DE GPU - Aller dans Runtime → Change runtime type → GPU")

## 2️⃣ Test du Dataset

In [ ]:
# Test rapide du dataloader
!python src/datasets/unpaired.py

### ⚠️ Si le test échoue

**Erreur probable:** Structure du dataset différente

**Solution:** Inspecter le dataset et adapter

In [ ]:
# 🔍 DÉBUG - Inspecter le dataset si test a échoué
from datasets import load_dataset

print("Chargement du dataset...")
dataset = load_dataset("huggan/night2day")

print("\n📊 Structure du dataset:")
print(dataset)

print("\n🔑 Clés disponibles:")
print(dataset.keys())

if 'train' in dataset:
    print("\n📝 Premier exemple:")
    print(dataset['train'][0].keys())
    print("\n📸 Type d'image:")
    print(type(dataset['train'][0]['image']))
else:
    print("\n⚠️ Pas de split 'train', splits disponibles:", list(dataset.keys()))

## 3️⃣ Configuration

In [ ]:
# Afficher la configuration actuelle
!cat src/configs/night2day.yaml

In [ ]:
# 🔧 AJUSTER LA CONFIG POUR COLAB (optionnel)
import yaml

# Charger config
with open('src/configs/night2day.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Ajustements pour Colab (GPU T4 15GB)
config['training']['epochs'] = 50  # Réduire si manque de temps
config['training']['batch_size'] = 4  # OK pour T4
config['hardware']['num_workers'] = 2  # Colab aime pas trop de workers

# Sauvegarder
with open('src/configs/night2day.yaml', 'w') as f:
    yaml.dump(config, f)

print("✅ Config ajustée pour Colab")
print(f"Epochs: {config['training']['epochs']}")
print(f"Batch size: {config['training']['batch_size']}")

## 4️⃣ Entraînement 🚀

**⏱️ Temps estimé:** 
- 50 epochs: ~3-4h sur T4
- 100 epochs: ~6-8h sur T4

**💡 Conseil:** Lancer et laisser tourner

In [ ]:
# Lancer l'entraînement
!python src/train.py --config src/configs/night2day.yaml

### 🔍 Monitoring (optionnel)

Si l'entraînement tourne, vous pouvez surveiller dans une autre cellule:

In [ ]:
# Vérifier les checkpoints sauvegardés
!ls -lh src/runs/night2day_cyclegan/

## 5️⃣ Évaluation et Samples

In [ ]:
# Générer des samples de test
!python src/eval.py --config src/configs/night2day.yaml \
                     --weights src/runs/night2day_cyclegan/best.pt

In [ ]:
# Visualiser quelques samples
import matplotlib.pyplot as plt
from PIL import Image
import os

samples_dir = 'src/runs/night2day_cyclegan/samples'
sample_files = sorted([f for f in os.listdir(samples_dir) if f.endswith('.png')])[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, fname in enumerate(sample_files):
    img = Image.open(os.path.join(samples_dir, fname))
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'Sample {i+1}', fontsize=10)

plt.tight_layout()
plt.savefig('samples_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Visualisé {len(sample_files)} samples")

## 6️⃣ Téléchargement du checkpoint

**Important:** Télécharger le fichier `best.pt` pour l'utiliser sur Jetson !

In [ ]:
# Compresser les résultats importants
!zip -r results.zip \
    src/runs/night2day_cyclegan/best.pt \
    src/runs/night2day_cyclegan/last.pt \
    src/runs/night2day_cyclegan/samples/ \
    src/runs/night2day_cyclegan/config.yaml \
    samples_overview.png

print("\n✅ Archive créée: results.zip")
!ls -lh results.zip

In [ ]:
# Télécharger l'archive
from google.colab import files

files.download('results.zip')
print("\n📥 Téléchargement lancé !")
print("\nContenu:")
print("  - best.pt (meilleur checkpoint)")
print("  - last.pt (dernier checkpoint)")
print("  - samples/ (images générées)")
print("  - config.yaml (configuration utilisée)")
print("  - samples_overview.png (aperçu)")

## 7️⃣ Statistiques finales

In [ ]:
# Afficher les statistiques du training
import json

# Taille des checkpoints
import os
best_size = os.path.getsize('src/runs/night2day_cyclegan/best.pt') / 1024**2
print(f"📊 Taille checkpoint: {best_size:.1f} MB")

# Nombre de samples générés
n_samples = len([f for f in os.listdir('src/runs/night2day_cyclegan/samples') if f.endswith('.png')])
print(f"📸 Samples générés: {n_samples}")

# Lire les métriques si disponibles
metrics_file = 'src/runs/night2day_cyclegan/test_metrics.json'
if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    print("\n📈 Métriques:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

print("\n✅ Training terminé !")

---

## 🎯 Prochaines étapes

1. ✅ **Télécharger** `results.zip`
2. ✅ **Décompresser** et récupérer `best.pt`
3. ✅ **Transférer** `best.pt` vers Jetson
4. ✅ **Lancer** le demo:
   ```bash
   python3 src/demo_live_split.py --weights best.pt --size 256
   ```

---

## 📝 Notes importantes

- **Colab gratuit:** Session limitée à 12h, sauvegarder régulièrement
- **GPU T4:** ~15GB VRAM, largement suffisant
- **Checkpoint:** Le fichier `best.pt` fait ~15-20 MB
- **Samples:** Vérifier visuellement la qualité

---

**Bon training ! 🚀**